<h1 align="center"> Data Preprocessing </h1>

<br>
<i> Import necessary libraries: <i>

In [1]:
# Additional imports needed to load src.model and src.dataset
import sys
sys.path.append("..")

import os
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
import pandas as pd
import numpy as np

from src.optimise import set_seed, capitalize_j, format_case_id

np.random.seed(2)

# Set random seed for weights
random_seed = 2
set_seed(random_seed)

Random seed set as 2


## Load Data: Radiomics and RNA

In [2]:
# Step 1: Load RNA-Seq data
df_rna = pd.read_csv("/home/kryan24/MRes_Ultrasound/data/rna_counts/RNA_counts_141024.csv", header=None, low_memory=False)
df_rna_transposed = df_rna.transpose()
new_header = df_rna_transposed.iloc[1]
df_rna_transposed = df_rna_transposed[2:]
df_rna_transposed.columns = new_header
df_rna_transposed.rename(columns=lambda x: x.strip(), inplace=True) # Trim whitespaces

# Step 3: Filter RNA data for 'ENSG' columns
rna_cols = df_rna_transposed.filter(like='ENSG')

# Step 4: Load and merge image data
df_images = pd.read_excel("/home/kryan24/MRes_Ultrasound/data/metadata/RNA_Sample_Linking2.xlsx")
df_images.rename(columns=lambda x: x.strip(), inplace=True) # Trim whitespaces
df_images['Sample ID'] = df_images['Sample ID'].str.replace(' ', '_')
df_images["SURGICAL"] = df_images["SURGICAL"].str.strip() # Remove whitespaces from SURGICAL

df_merged = pd.merge(df_images, df_rna_transposed, left_on="Sample ID", right_on="Geneid")

df_merged_copy = df_merged

# Step 5: Correct data quality issues
df_merged["Extraction site"] = df_merged["Extraction site"].str.strip()
df_merged["Extraction site"] = df_merged["Extraction site"].str.lower()

# Step 6: Read the CSV file into a DataFrame 
df_radiomics = pd.read_csv("/home/kryan24/MRes_Ultrasound/data/metadata/radiomics_clinical_Barcroft_12_01_23.csv", low_memory=False)
# Remove first 2 columns
df_radiomics.drop(df_radiomics.columns[[0, 1]], axis=1, inplace=True)

# Filter columns
df_radiomics = df_radiomics.drop(df_radiomics.iloc[:, 3907:3950], axis=1)
df_radiomics["MAHCINE_CORECCTED"] = df_radiomics["MAHCINE_CORECCTED"].str.strip() # Remove whitespaces from MAHCINE_CORECCTED

# Step 7: Match radiomics data to metadata and filter DataFrame
df_radiomics_meta = df_merged[["Extraction site", "SURGICAL", "CASE ID", "CASE ID2"]]

# Step 8: Merge DataFrames
df_merged2 = pd.merge(df_radiomics_meta, df_radiomics, left_on="CASE ID", right_on="ID")

# Step 9: Correct data quality issues
df_merged2 = df_merged2.drop(["ID"], axis=1)

# Step 10: Keep the same samples for RNA as radiomics data
case_id_list = df_merged2["CASE ID"].tolist()
mask = df_merged["CASE ID"].isin(case_id_list)
df_merged = df_merged[mask]

In [3]:
# Check length of both paired sample dataframes
print("Number of rows in paired radiomics samples:", df_merged2.shape[0])
print("Number of rows in paired RNA samples:", df_merged.shape[0])

Number of rows in paired radiomics samples: 36
Number of rows in paired RNA samples: 36


### Pair Images and RNA Data

In [4]:
# Filter radiomics data 
rad_cols = df_merged2.drop(["Extraction site", "SURGICAL", "CASE ID", "CASE ID2", "MAHCINE_CORECCTED"], axis=1)

# Step 1: Prepare image filenames and filter data
image_directory = "/home/kryan24/MRes_Ultrasound/CLIPRNA/images_matched"
image_filenames_rna = []
image_filenames_rad = []
rna_data_filtered = []
rad_data_filtered = []
surgical_rna = []
surgical_rad = []
sample_count_rna = 0 
sample_count_rad = 0 

rna_paired_id_list = []
rad_paired_id_list = [] 

In [5]:
for index, row in df_merged.iterrows():

    # Format and capitalize the CASE IDs
    case_id = capitalize_j(format_case_id(row['CASE ID']))
    case_id2 = capitalize_j(format_case_id(row['CASE ID2']))

    for file_name in os.listdir(image_directory):
            if file_name.endswith('.nii.gz') and 'seg' not in file_name:
                file_name_corrected = capitalize_j(file_name)  # Capitalize 'j' in the filename

                # Check if either case_id or case_id2 is a substring of the filename
                # Only check if the ID is non-empty
                if (case_id and case_id in file_name_corrected):
                    image_filenames_rna.append(file_name)
                    rna_data_filtered.append(
                        row[rna_cols.columns].apply(pd.to_numeric, errors='coerce').fillna(0)
                    )
                    surgical_rna.append(row["SURGICAL"])
                    rna_paired_id_list.append(case_id)
                    sample_count_rna += 1
                    break  # Only use the first matching image for each row

                if (case_id2 and case_id2 in file_name_corrected):
                    image_filenames_rna.append(file_name)
                    rna_data_filtered.append(
                        row[rna_cols.columns].apply(pd.to_numeric, errors='coerce').fillna(0)
                    )
                    surgical_rna.append(row["SURGICAL"])
                    rna_paired_id_list.append(case_id2)
                    sample_count_rna += 1
                    break  # Only use the first matching image for each row

print(f"Number of Samples: ", sample_count_rna)

Number of Samples:  29


In [6]:
for index, row in df_merged2.iterrows():

    # Format and capitalize the CASE IDs
    case_id = capitalize_j(format_case_id(row['CASE ID']))
    case_id2 = capitalize_j(format_case_id(row['CASE ID2']))

    for file_name in os.listdir(image_directory):
            if file_name.endswith('.nii.gz') and 'seg' not in file_name:
                file_name_corrected = capitalize_j(file_name)  # Capitalize 'j' in the filename

                # Check if either case_id or case_id2 is a substring of the filename
                # Only check if the ID is non-empty
                if (case_id and case_id in file_name_corrected):
                    image_filenames_rad.append(file_name)
                    rad_data_filtered.append(
                        row[rad_cols.columns].apply(pd.to_numeric, errors='coerce').fillna(0)
                    )
                    surgical_rad.append(row["SURGICAL"])
                    rad_paired_id_list.append(case_id)
                    sample_count_rad += 1
                    break  # Only use the first matching image for each row

                if (case_id2 and case_id2 in file_name_corrected):
                    image_filenames_rad.append(file_name)
                    rad_data_filtered.append(
                        row[rad_cols.columns].apply(pd.to_numeric, errors='coerce').fillna(0)
                    )
                    surgical_rad.append(row["SURGICAL"])
                    rad_paired_id_list.append(case_id2)
                    sample_count_rad += 1
                    break  # Only use the first matching image for each row

print(f"Number of Samples: ", sample_count_rad)

Number of Samples:  29


In [7]:
flag = (surgical_rna == surgical_rad)
print(f"Matching Diagnoses: ", flag)

flag2 = (image_filenames_rna == image_filenames_rad)
print(f"Matching Images: ", flag2)

flag3 = (rna_paired_id_list == rad_paired_id_list)
print(f"Matching IDs: ", flag3)

Matching Diagnoses:  True
Matching Images:  True
Matching IDs:  True


In [8]:
# Create DataFrame for surgical matching
match_dict = {"image_name": image_filenames_rad, "ID": rad_paired_id_list, "SURGICAL": surgical_rad} 
match_df = pd.DataFrame(match_dict)
match_df

,image_name,ID,SURGICAL
0,12930_3.nii.gz,12930,B-S
1,13291_0010.nii.gz,13291,M-S
2,14023_3.nii.gz,14023,M-S
3,14056_3.nii.gz,14056,M-S
4,14654_3.nii.gz,14654,M-S
5,14024_2.nii.gz,14024,M-S
6,14984_0039.nii.gz,14984,M-S
7,16106_0022.nii.gz,16106,B-S
8,16134_0049.nii.gz,16134,M-S
9,16208_1.nii.gz,16208,B-S


### Get Unpaired Images and RNA Data 
(_validation_)

In [9]:
# Step 6: Read the CSV file into a DataFrame 
df_radiomics = pd.read_csv("/home/kryan24/MRes_Ultrasound/data/metadata/radiomics_clinical_Barcroft_12_01_23.csv", low_memory=False)
# Remove first 2 columns
df_radiomics.drop(df_radiomics.columns[[0, 1]], axis=1, inplace=True)
# Filter columns
surg_placeholder = df_radiomics["SURGICAL"]
df_radiomics = df_radiomics.drop(df_radiomics.iloc[:, 3907:3950], axis=1)
df_radiomics["MAHCINE_CORECCTED"] = df_radiomics["MAHCINE_CORECCTED"].str.strip() # Remove whitespaces from MAHCINE_CORECCTED
df_radiomics.insert(0, "SURGICAL", surg_placeholder)
df_radiomics["SURGICAL"] = df_radiomics["SURGICAL"].str.strip() 
df_radiomics["SURGICAL"] = df_radiomics["SURGICAL"].replace(["B-NS", "B-NS -->THEATRE"], "B-S")
df_radiomics["SURGICAL"] = df_radiomics["SURGICAL"].replace(["M-NS"], "M-S")
df_radiomics = df_radiomics.loc[df_radiomics["SURGICAL"] != "NS"]
df_radiomics["ID"] = df_radiomics["ID"].astype(str)

In [10]:
rad_paired_id_list_noj = list(map(lambda x: "26246" if x == "J630" else x, rad_paired_id_list))
df_unpaired_rad = df_radiomics[~df_radiomics["ID"].isin(rad_paired_id_list_noj)]
df_unpaired_rna = df_merged_copy[~df_merged_copy["CASE ID"].isin(rna_paired_id_list) & ~df_merged_copy["CASE ID2"].isin(rna_paired_id_list)]


In [11]:
# Check length of both unpaired sample dataframes
print("Number of rows in unpaired radiomics samples:", df_unpaired_rad.shape[0])
print("Number of rows in unpaired RNA samples:", df_unpaired_rna.shape[0])

Number of rows in unpaired radiomics samples: 547
Number of rows in unpaired RNA samples: 38


In [12]:
check1 = df_unpaired_rad["ID"].to_list()
check2 = rad_paired_id_list
set(check1) & set(check2)

set()

In [13]:
check1 = df_unpaired_rna["CASE ID"].to_list() + df_unpaired_rna["CASE ID2"].to_list()
check2 = rna_paired_id_list
set(check1) & set(check2)

set()

In [40]:
# Create DataFrame for surgical matching
match_rad_dict = {"CASE ID": df_unpaired_rad["ID"].astype("int64"), "SURGICAL": df_unpaired_rad["SURGICAL"]} 
match_rad_df = pd.DataFrame(match_rad_dict)
match_rad_df

,CASE ID,SURGICAL
0,11935,B-S
1,12041,B-S
2,12165,B-S
3,12287,M-S
4,12665,M-S
...,...,...
572,756,B-S
573,757,B-S
574,759,B-S
575,760,M-S


In [20]:
# Create DataFrame for surgical matching
match_rna_dict = {"CASE ID": df_unpaired_rna["CASE ID"], "CASE ID2": df_unpaired_rna["CASE ID2"], "SURGICAL": df_unpaired_rna["SURGICAL"]} 
match_rna_df = pd.DataFrame(match_rna_dict)
match_rna_df

,CASE ID,CASE ID2,SURGICAL
0,12930,NaN,B-S
1,13291,NaN,M-S
2,13750,NaN,M-S
3,14023,NaN,M-S
4,14056,NaN,M-S
5,14654,NaN,M-S
6,14024,NaN,M-S
7,14984,NaN,M-S
8,16106,NaN,B-S
9,16134,NaN,M-S


## Load Data: Image and RNA-seq Embeddings
(same as above)

### Get Unpaired Images 
(_validation_)

In [41]:
df_images_ids = df_images[["SURGICAL", "CASE ID", "CASE ID2"]]
df_images_ids = df_images_ids.drop_duplicates(keep="first")
df_images_ids

,SURGICAL,CASE ID,CASE ID2
0,M-S,12287,NaN
3,B-S,12930,NaN
9,M-S,13291,NaN
13,M-S,13750,NaN
18,M-S,14023,NaN
21,M-S,14056,NaN
23,M-S,14654,NaN
24,M-S,14024,NaN
26,M-S,14984,NaN
30,B-S,15346,NaN


In [83]:
df_images_merged = pd.merge(match_rad_df, df_images_ids, how='outer', on=['CASE ID'])
df_images_merged["SURGICAL_x"] = df_images_merged["SURGICAL_x"].fillna(df_images_merged["SURGICAL_y"])
df_images_merged = df_images_merged.drop("SURGICAL_y", axis="columns")
df_images_merged = df_images_merged.rename(columns={"SURGICAL_x": "SURGICAL"})
df_images_merged = df_images_merged[["CASE ID", "CASE ID2", "SURGICAL"]]
merged_id_list = match_rad_df["CASE ID"].tolist()
mask_merged = df_images_merged["CASE ID"].isin(merged_id_list)
df_images_merged = df_images_merged[mask_merged]
df_images_merged = df_images_merged.drop_duplicates(subset="CASE ID", keep="first")

In [85]:
df_images_merged

,CASE ID,CASE ID2,SURGICAL
0,430,NaN,B-S
1,472,NaN,B-S
2,479,NaN,B-S
3,491,NaN,B-S
4,494,NaN,B-S
...,...,...,...
574,26624,J567,M-S
575,26631,NaN,B-S
576,26633,NaN,M-S
577,26636,NaN,M-S


In [96]:
# Step 11: Obtain unpaired data for validation
unpaired_image_filenames = []
surgical_unpaired = []
number_unpaired_images = 0 
unpaired_image_id_list = []

In [97]:
for index, row in df_images_merged.iterrows():

    # Format and capitalize the CASE IDs
    case_id = capitalize_j(format_case_id(row['CASE ID']))
    case_id2 = capitalize_j(format_case_id(row['CASE ID2']))

    for file_name in os.listdir(image_directory):
        if file_name.endswith('.nii.gz') and 'seg' not in file_name:
            file_name_corrected = capitalize_j(file_name)  # Capitalize 'j' in the filename

            # Check if either case_id or case_id2 is a substring of the filename
            # Only check if the ID is non-empty
            if (case_id and case_id in file_name_corrected):
                unpaired_image_filenames.append(file_name)
                surgical_unpaired.append(row["SURGICAL"])
                unpaired_image_id_list.append(case_id)
                number_unpaired_images += 1
                break

            if (case_id2 and case_id2 in file_name_corrected):
                unpaired_image_filenames.append(file_name)
                surgical_unpaired.append(row["SURGICAL"])
                unpaired_image_id_list.append(case_id2)
                number_unpaired_images += 1
                break

print(f"Number of Unpaired Image Samples: ", number_unpaired_images)

Number of Unpaired Image Samples:  408


In [102]:
match_image_list = list(set(unpaired_image_filenames) & set(image_filenames_rad))
match_image_list

['J630_0008.nii.gz',
 '14654_3.nii.gz',
 '26417_1.nii.gz',
 '25962_0006.nii.gz',
 '16134_0049.nii.gz',
 '25607_0005.nii.gz']

In [105]:
# Create DataFrame for surgical matching
match_img_dict = {"image_name": unpaired_image_filenames, "CASE ID": unpaired_image_id_list, "SURGICAL": surgical_unpaired} 
match_img_df = pd.DataFrame(match_img_dict)
match_img_df = match_img_df[~match_img_df["image_name"].isin(match_image_list)]
match_img_df

,image_name,CASE ID,SURGICAL
0,j472_1.nii.gz,472,B-S
1,j479_3.nii.gz,479,B-S
2,j491_7.nii.gz,491,B-S
3,j494_5.nii.gz,494,B-S
4,j518_6.nii.gz,518,B-S
...,...,...,...
403,26606_0015.nii.gz,26606,B-S
404,26607_0014.nii.gz,26607,B-S
405,26608_0007.nii.gz,26608,B-S
406,26609_0018.nii.gz,26609,B-S


In [106]:
match_img_df["SURGICAL"].value_counts()

SURGICAL
B-S    327
M-S     75
Name: count, dtype: int64